In [0]:
# Set default catalog and schema
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")
spark.sql(f"USE SCHEMA {schema}")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.{schema}.bakehouse_product_insights AS
SELECT
    t.product,
    COUNT(t.transactionID)              AS times_ordered,
    SUM(t.quantity)                     AS total_units_sold,
    ROUND(SUM(t.totalPrice), 2)         AS total_revenue,
    ROUND(AVG(t.totalPrice), 2)         AS avg_transaction_value,
    COUNT(DISTINCT t.franchiseID)       AS franchises_selling,
    ROUND(
        SUM(t.totalPrice) * 100.0 / SUM(SUM(t.totalPrice)) OVER (),
        2
    )                                   AS pct_of_total_revenue,
    CASE
        WHEN SUM(t.quantity) >= 1000 THEN 'Best Seller'
        WHEN SUM(t.quantity) >= 500  THEN 'Popular'
        ELSE 'Niche'
    END                                 AS popularity_tier
FROM samples.bakehouse.sales_transactions t
GROUP BY t.product
ORDER BY total_revenue DESC;   
""")